In [ ]:
import uuid
import pandas as pd
import geopandas as gpd
import os
from os.path import join
import sys
from pathlib import Path
from tqdm import tqdm
import json

# to have progress bar in the notebook
tqdm.pandas()
with open('dataproduction_config.json') as f:
    DATA_CONFIG = json.load(f)

# to retrieve the utils function used by all notebooks
parent_dir = os.path.abspath('../../../')
if parent_dir not in sys.path: sys.path.insert(0, parent_dir)
from utils.data_modeling import *
from utils.rde import RDE


DATA_SRC_PATH = Path(join(parent_dir, 'data-lausanne/1831-cadastre-berney/'))

# aribtrary namespace, just to generate reproducible UUIDv5 from the entries of the dataset.
VTM_UUID5_NS = uuid.uuid5(uuid.NAMESPACE_URL, DATA_CONFIG['UUID_NAMESPACE'])
DS_SLUG = DATA_CONFIG['DATASET_CONFIGURATION']['slug']
DS_UUID = str(uuid.uuid5(VTM_UUID5_NS, DS_SLUG))

lausanne_area_uuid = get_single_object_uuid(DATA_CONFIG['AREA_FILE_LOC'])

DS_OBJ = (DS_UUID, DS_SLUG)
TR_OBJ = (datetime_obj_from_int_time(DATA_CONFIG['TIMERANGE_MINIMUM']), datetime_obj_from_int_time(DATA_CONFIG['TIMERANGE_MAXIMUM'], match_to_end=True))
DATA_FOLDER = 'data'

# Geometry RDE production
geometries_fp = join(DATA_SRC_PATH, 'Berney_merge_legende_v7-7_formatted_for_timeatlas.geojson')
wh_fp = join(DATA_SRC_PATH, 'wh_images.csv')
# sample for testing uuid_gen
gdf = gpd.read_file(geometries_fp, use_arrow=True).to_crs("EPSG:4326")


gdf['start_time'] = pd.Series(data = [TR_OBJ[0]] * len(gdf), name='start_time')
gdf['end_time'] = pd.Series(data = [TR_OBJ[1]] * len(gdf), name='end_time')

cadaster_layer_uuid = get_layer_uuid(get_filepath_like('../../maps/lausanne-1831-berney/layers', 'json'), 'cadaster')
tqdm.pandas(desc="Generating uuid from geometry")
gdf['uuid'] = gdf.progress_apply(lambda row: make_uuid_from_row_selection(VTM_UUID5_NS, row, ['geometry']), axis=1)

# entries without identifier only relate to geometric features without information from the registry, they don't make sense as Historical Record
df = gdf[gdf['identifier'] != ''].copy()
df = df.groupby('identifier').first().reset_index()

from shapely.ops import unary_union
from shapely.geometry import Point
id_geoms = df.groupby('identifier').agg({'uuid': lambda x: list(x), 'geometry': lambda x: list(x)}).reset_index()
def compute_center(geoms):
    # geoms is a list of shapely geometry objects
    if len(geoms) == 0:
        return None
    if len(geoms) == 1:
        # single geometry: use centroid
        c = geoms[0].centroid
    else:
        # multiple geometries: fuse them and take centroid
        fused = unary_union(geoms)
        c = fused.centroid
    # return as (x, y) tuple
    return Point(c.x, c.y)
id_geoms['center'] = id_geoms['geometry'].apply(compute_center)

df['has_geometry'] = df['identifier'].apply(lambda idf: id_geoms.loc[id_geoms['identifier'] == idf, 'uuid'].values[0])
df['center'] = df['identifier'].apply(lambda idf: id_geoms.loc[id_geoms['identifier'] == idf, 'center'].values[0])
df = df.drop(columns=['uuid'])

gdf['layer_uuid'] = cadaster_layer_uuid
gdf['rde_type'] = "geometry"
QA_check_uuid_are_unique(gdf)

if not QA_check_all_geometries_are_valid(gdf, raise_exception=False):
    from shapely.validation import make_valid
    gdf['geometry'] = gdf['geometry'].apply(lambda g: g if g.is_valid else make_valid(g))
    QA_check_all_geometries_are_valid(gdf)

geom_shorthand = 'lausanne_1831_berney_geometries'
# "parcel_type" was removed for consistency with the other datasets. 
save_data_file_if_different(DATA_FOLDER, 'geometries', gdf[['uuid', 'geometry', 'start_time', 'end_time', 'layer_uuid', 'rde_type']], geom_shorthand, RDE.GEOM.value)
tqdm.pandas(desc="Generating uuid for hr")
df['hr_uuid'] = df.progress_apply(lambda v: make_uuid_from_row_selection(VTM_UUID5_NS, v, ['identifier'], ad_hoc_seed='hr'), axis=1)

tqdm.pandas(desc="Generating uuid for obs")
df['obs_uuid'] = df.progress_apply(lambda v: make_uuid_from_row_selection(VTM_UUID5_NS, v, ['center'], ad_hoc_seed='obs'), axis=1)

tqdm.pandas(desc="Generating uuid for poi")
df['poi_uuid'] = df.progress_apply(lambda v: make_uuid_from_row_selection(VTM_UUID5_NS, v, ['center'], ad_hoc_seed='poi'), axis=1)

In [ ]:

obs_df = df[['obs_uuid','hr_uuid', 'poi_uuid', 'center', 'has_geometry']].copy().reset_index().set_index('obs_uuid')
obs_df['poi_uuid'] = obs_df['poi_uuid'].apply(lambda v: [v])
# I have to do that because there is 7 obs. that have two historical sources recording it...
tpe = 'parcel ownership'
obs_from_row = lambda v: produce_obs_obj(v.obs_uuid, TR_OBJ, DS_UUID, v.hr_uuid, tpe, v.center, v.has_geometry, v.poi_uuid)
obs = [obs_from_row(v) for _, v in obs_df.reset_index().iterrows()]
gdf_obs = gpd.GeoDataFrame(obs)
gdf_obs = gdf_obs.set_geometry('coordinate').set_crs('EPSG:4326').set_index('uuid')
# when the geodataframe is serialized, the label of the geometry column is lost (default to geometry), doing it here makes it explicit and make the save_data_file_if_different work.
gdf_obs = gdf_obs.rename(columns={'coordinate': 'geometry'})
gdf_obs = gdf_obs.set_geometry('geometry')
QA_check_uuid_are_unique(gdf_obs.reset_index())
obs_shorthand = 'lausanne_1831_berney_obs'
save_data_file_if_different(DATA_FOLDER, 'observations', gdf_obs, obs_shorthand, RDE.OBS.value)
QA_check_unique_uuid_in_uuid_array(gdf_obs.reset_index(), 'has_geometry')


# POI RDE Production
df_poi = df[['center', 'poi_uuid', 'obs_uuid']].groupby(by=['poi_uuid']).agg(list)

gdf_poi = gpd.GeoDataFrame([produce_poi_obj(v.poi_uuid, v.center[0], v.obs_uuid) for _, v in df_poi.reset_index().iterrows()])
gdf_poi = gdf_poi.set_geometry('coordinate').set_crs('EPSG:4326').set_index('uuid')
# when the geodataframe is serialized, the label of the geometry column is lost (default to geometry), doing it here makes it explicit and make the save_data_file_if_different work.
gdf_poi = gdf_poi.rename(columns={'coordinate': 'geometry'})
gdf_poi = gdf_poi.set_geometry('geometry')
save_data_file_if_different(DATA_FOLDER, 'points_of_interest', gdf_poi, 'lausanne_1831_berney_pois', RDE.POI.value)
QA_check_uuid_are_unique(gdf_poi.reset_index())

QA_check_unique_uuid_in_uuid_array(gdf_poi.reset_index(), 'represents')

In [ ]:
#HR RDE Production
exclude_hr_labels = {
    'geometry_id', 
    'has_geometry', 
    'poi_uuid',
    'obs_uuid',
    'hr_uuid',
    'geometry'
}

drop_cols = {
    #TODO
}

exlude_cols = exclude_hr_labels.union(drop_cols)

df = df.replace({np.nan:None})
hr_metadata_cols = list(set(df.columns).difference(exlude_cols))
hr_metadata_cols

In [ ]:
tpe = 'cadaster registry'
recs = [produce_hr_obj(r.hr_uuid,\
                       DS_UUID,\
                       [[r['obs_uuid'], 'identifier']],\
                       TR_OBJ,\
                       tpe,\
                       r[hr_metadata_cols].to_dict()) \
            for _, r in df.iterrows()
        ]

hr_shorthand = 'lausanne_1831_berney_historical_records'
save_data_file_if_different(DATA_FOLDER, 'historical_records',recs, hr_shorthand, RDE.HR.value)

df_of_hr = pd.DataFrame(data = recs)
QA_check_uuid_are_unique(df_of_hr)
QA_check_unique_uuid_in_uuid_array(df_of_hr, 'documents')

In [ ]:
# normally the pages are already sorted as is in the CSV
df_pages = pd.read_csv(wh_fp)

def format_filename_to_code(filename:str):
    code = filename.replace('PC_1827-1831_Berney_Vol-', '')
    if 'legende' in code:
        code = code.replace('_legende', '') + (' (légende)')
    code = code.split('_')[-1].replace('.jpg', '')
    if code.isnumeric():
        code = int(code)
    return code

df_pages['page_num'] = df_pages['filename'].apply(lambda x: format_filename_to_code(x))

In [ ]:

IMG_LOC_PREFIX = 'lausanne/berney/'

manifest_uid = str(uuid.uuid5(VTM_UUID5_NS, f'manifest_{DS_SLUG}'))
df_pages['page_obj'] = None
for i, x in df_pages.iterrows():
    df_pages.at[i, 'page_obj'] = iiif.generate_page_object(VTM_UUID5_NS, DS_SLUG, x['page_num'], manifest_uid, \
                                                                'pg. '+str(x['page_num']), IMG_LOC_PREFIX+x['filename'],\
                                                                x['media_type'], x['width'], x['height'], 'it')
    
df_pages['canvas_id'] = df_pages['page_obj'].apply(lambda x: x['id'])
df_of_hr['page'] = df_of_hr['annotated_content'].apply(lambda v: v['PAGE_NUM'])
page_to_canvas = df_pages[['page_num', 'canvas_id']].set_index('page_num').to_dict()['canvas_id']
# preparing the data to insert in the canvas of the manifest.
df_iiif_links = df_of_hr[df_of_hr['page'].notnull()]


collection = {}
registry_label = {
                    "en": [f'Commercial guide of Venice from 1857'],
                    "it": [f'Guida commerciale di Venezia del 1857'],
                    "fr": [f'Annuaire commercial de Venise de 1857']
                }

collection[manifest_uid] = (registry_label, df_pages['page_obj'].tolist()[0])

collection_manifest_uid = str(uuid.uuid5(VTM_UUID5_NS, f'collection_{DS_SLUG}'))
with open(f'data/iiif/collections/{collection_manifest_uid}.json', 'w+', encoding='utf-8') as f:
    json.dump(iiif.generate_collection_manifest(collection_manifest_uid, registry_label, collection), f, indent=2, ensure_ascii=False)

col_in_order = ['LAST_N', 'FIRST_N', 'PER_GRP', 'PER_COMPL', 'LOC_PAR', 'LOC_STR', 'NUM', 'LOC_COMPL']

def cg_1857_metadata_object_to_string_representation(metadata: dict) -> str:
    quick_check = lambda x: str(x) if type(x) is int else x if type(x) is str and x.lower() != 'nan' and len(x) > 0 else ''
    vals = [quick_check(metadata.get(v, '')) for v in col_in_order]
    return ' '.join([v for v in vals if len(v) > 0])

df_iiif_links['iiif_display_string'] = df_iiif_links['annotated_content'].apply(cg_1857_metadata_object_to_string_representation)
df_iiif_links['iiif_metadata_obj'] = df_iiif_links.apply(lambda x: (x['uuid'], x['iiif_display_string']), axis=1)
# applying the page to canvas mapping.
df_iiif_links['canvas_id'] = df_iiif_links['page'].apply(lambda v: page_to_canvas.get(v, None))
# # the hr_uuid is missing. 
iiif_links = df_iiif_links[['canvas_id', 'iiif_metadata_obj']].groupby('canvas_id', sort=False).agg(list).reset_index().set_index('canvas_id')['iiif_metadata_obj'].to_dict()
df_pages['page_obj'] = df_pages['page_obj'].apply(lambda x: dict(x, metadata = iiif_links.get(x['id'], '')))

with open(f'data/iiif/manifests/{manifest_uid}.json', 'w+', encoding='utf-8') as f:
    json.dump(iiif.generate_manifest_object(VTM_UUID5_NS, manifest_uid, registry_label, 'en', df_pages['page_obj'].tolist()), f, indent=2, ensure_ascii=False)



In [ ]:
gdf_obs

In [ ]:


# Dataset RDE Production
CONF = DATA_CONFIG['DATASET_CONFIGURATION']

filtered_df = df[hr_metadata_cols].copy()
remaining_vals = list(filtered_df.columns)
order = CONF['labels'].keys()

ds_conf = produce_configuration_file_from_metadata_df(
    VTM_UUID5_NS, 
    filtered_df[order], CONF["indexed"], 
    CONF["short_display"],
    CONF["hidden"], 
    {}, 
    CONF["tagged_fields"], 
    CONF["labels"],
    main_label=CONF["main_label"], 
    sub_label=CONF["sub_label"]
)

ds = produce_dataset_obj(
    DS_UUID,
    DS_SLUG,
    '1.0',
    CONF['name'],
    CONF['description'],
    CONF['paradata'],
    [],
    TR_OBJ,
    0,
    ds_conf,
    [lausanne_area_uuid],
    publish_obj=(CONF['doi'], CONF['github_link'])
)

save_data_file_if_different(DATA_FOLDER,'datasets', [ds], 'lausanne_1831_berney_dataset', RDE.DATASET.value, is_dataset_obj=True)